In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
print("Folders inside Kaggle Input:")
print(os.listdir("/kaggle/input/"))

In [ ]:
import os

print("--- CLOUD PATH DIAGNOSTIC ---")
if os.path.exists("/kaggle/input/tcga-coad-msi-mss-jpg"):
    print("✅ Images found in Kaggle Input!")
else:
    print("❌ Images NOT found. (Check the 'input' folder name in the right sidebar)")

if os.path.exists("/kaggle/working/patient_registry.json"):
    print("✅ JSON Contract found in Working directory!")
else:
    print("❌ JSON NOT found. Please upload it using the right sidebar.")

In [ ]:
%pwd

In [ ]:
pwd

In [ ]:
ls

In [ ]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
from tqdm import tqdm

# --- Configuration ---
BASE_DATA_DIR = "/kaggle/input/datasets/joangibert/tcga_coad_msi_mss_jpg" 
REGISTRY_PATH = "/kaggle/input/datasets/moki27/patient-registry/patient_registry.json"
OUTPUT_DIR = "/kaggle/working/patient_vision_tensors" 

# 🚀 BUMPED: Quadrupled to saturate 2x 16GB GPUs
BATCH_SIZE = 256 

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Hardware Check & Multi-GPU Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Initializing Extraction on Device: {device}")

# 2. Load Model & Processor
print("Loading Phikon-v2 (DINOv2 Architecture)...")
processor = AutoImageProcessor.from_pretrained("owkin/phikon-v2")
model = AutoModel.from_pretrained("owkin/phikon-v2")

# 🚀 NEW: Engage Dual-GPU DataParallel
if torch.cuda.device_count() > 1:
    print(f"🔥 Engaging {torch.cuda.device_count()} GPUs for parallel processing!")
    model = torch.nn.DataParallel(model)

model = model.to(device)
model.eval() 

# 3. Custom PyTorch Dataset (Unchanged structurally)
class PatientBagDataset(Dataset):
    def __init__(self, image_paths, processor):
        self.image_paths = image_paths
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = os.path.join(BASE_DATA_DIR, self.image_paths[idx])
        try:
            image = Image.open(img_path).convert("RGB")
            pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)
            return pixel_values, img_path
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros((3, 224, 224)), img_path

# 4. The Extraction Loop
def extract_features():
    with open(REGISTRY_PATH, 'r') as f:
        registry = json.load(f)
        
    print(f"Starting extraction for {len(registry)} patients...")
    
    for patient_id, data in tqdm(registry.items(), desc="Patients Processed"):
        save_path = os.path.join(OUTPUT_DIR, f"{patient_id}_vision.pt")
        
        if os.path.exists(save_path):
            continue
            
        image_paths = data["image_paths"]
        
        if not image_paths:
            continue
            
        dataset = PatientBagDataset(image_paths, processor)
        
        # 🚀 NEW: Overdrive the DataLoader
        dataloader = DataLoader(
            dataset, 
            batch_size=BATCH_SIZE, 
            shuffle=False, 
            num_workers=4,           # Maximize Kaggle CPU threads
            pin_memory=True,         # Fast-lane transfer to GPU
            prefetch_factor=2        # Pre-load the next batch while GPU works
        )
        
        patient_features = []
        
        with torch.no_grad():
            for batch_pixels, _ in dataloader:
                batch_pixels = batch_pixels.to(device)
                
                outputs = model(pixel_values=batch_pixels)
                
                # Extract CLS token
                cls_features = outputs.last_hidden_state[:, 0, :]
                patient_features.append(cls_features.cpu())
                
        final_tensor = torch.cat(patient_features, dim=0)
        torch.save(final_tensor, save_path)

if __name__ == "__main__":
    extract_features()
    print("\n✅ Vision Extraction Complete! All patient tensors saved.")

In [ ]:
import shutil
import os

folder_to_zip = "/kaggle/working/patient_vision_tensors"
output_filename = "/kaggle/working/vision_features_backup" # Note: No .zip extension here

print(f"Compressing features from {folder_to_zip}...")
shutil.make_archive(output_filename, 'zip', folder_to_zip)
print(f"✅ Compression complete! File saved as: {output_filename}.zip")


In [ ]:
from IPython.display import FileLink
# This creates a direct download link in your notebook output
display(FileLink('/kaggle/working/vision_features_backup.zip'))

In [ ]:
import os

print("🚀 Establishing secure cloud bridge...")
print("Uploading 1GB file. This will take roughly 1 to 2 minutes. Do not close the tab.")

# We use cURL to push the file to transfer.sh (a standard developer file transfer tool)
os.system('curl --upload-file /kaggle/working/vision_features_backup.zip https://transfer.sh/vision_backup.zip > download_link.txt')

with open('download_link.txt', 'r') as f:
    url = f.read().strip()
    print("\n" + "="*50)
    print("✅ TRANSFER COMPLETE!")
    print(f"👉 CLICK HERE TO DOWNLOAD: {url}")
    print("="*50)

In [ ]:
from IPython.display import HTML

# Create a direct hyperlink to the local container file
download_button = """
<a href="./vision_features_backup.zip" download="vision_features_backup.zip" 
   style="background-color: #4CAF50; color: white; padding: 15px 32px; 
   text-align: center; text-decoration: none; display: inline-block; 
   font-size: 16px; margin: 4px 2px; cursor: pointer; border-radius: 8px;">
   📥 FORCE DOWNLOAD 1GB ZIP FILE
</a>
"""
display(HTML(download_button))